# HW2 — Data cleaning and validation
**Henri Mertens · Part 1, sections 1.1–1.3**

Run all cells in order from this homework directory, with a **local working copy** at `data/data.db`.
Sections 1.1–1.2 inspect the source through a read-only connection. Section 1.3 creates or replaces
only `ohlc_hw_cleaned` in that working database. Source tables remain unchanged.
Original values, source-row identifiers and repair/review flags are retained.
The full pre-repair diagnosis is available as `quality_report`; large displayed tables are limited.

In [1]:
from pathlib import Path
from functools import lru_cache
import hashlib
import html
import re
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 100)
DB_PATH = Path('data/data.db')
if not DB_PATH.is_file():
    raise FileNotFoundError('Run from the homework directory containing data/data.db.')

def file_sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

original_database_hash = file_sha256(DB_PATH)
connection = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
try:
    connection.execute('PRAGMA query_only = ON')
    raw = pd.read_sql_query('SELECT rowid AS source_row, * FROM ohlc_hw ORDER BY rowid', connection)
    reference = pd.read_sql_query('SELECT * FROM ohlc', connection)
    original_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", connection
    )['name'].tolist()
finally:
    connection.close()
print(f'Loaded {len(raw):,} dirty rows and {len(reference):,} reference rows, read-only.')

Loaded 397,711 dirty rows and 368,251 reference rows, read-only.


## 1.1 Load and inspect

| Column | Meaning |
|---|---|
| `ts` | Observation date |
| `ticker` | Stock identifier; some supplied labels contain presentation markup |
| `open`, `high`, `low`, `close` | Raw daily opening, highest, lowest and closing prices |
| `adj_close` | Vendor-adjusted closing price, on a different adjustment basis from raw OHLC |
| `volume` | Reported traded volume |
| `source_row` | Original SQLite row identifier added for this inspection, not a market-data field |

Raw OHLC bounds apply to open and close, not to adjusted close. Different stocks have different supplied histories.

In [2]:
raw.info()
display(raw.head())
display(raw.drop(columns='source_row').describe())
print('Raw date range:', raw['ts'].min(), 'to', raw['ts'].max())
print('Distinct raw ticker labels:', raw['ticker'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   source_row  397711 non-null  int64  
 1   ts          397711 non-null  str    
 2   open        397711 non-null  float64
 3   high        397711 non-null  float64
 4   low         397711 non-null  float64
 5   close       397711 non-null  float64
 6   adj_close   377856 non-null  float64
 7   volume      397711 non-null  int64  
 8   ticker      397711 non-null  str    
dtypes: float64(5), int64(2), str(2)
memory usage: 34.7 MB


,source_row,ts,open,high,low,close,adj_close,volume,ticker
0,1,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,2,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,3,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,4,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,5,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Raw date range: 1962-01-02 to 2024-12-06
Distinct raw ticker labels: 150


### Ticker normalization audit
Remove HTML tags, decode HTML entities, strip surrounding whitespace and standardize case **before** grouping.
Dots and hyphens in stock symbols remain valid. This creates an analysis copy; `raw` is unchanged.

In [3]:
PRICE_COLUMNS = ['open', 'high', 'low', 'close', 'adj_close']
RAW_PRICE_COLUMNS = ['open', 'high', 'low', 'close']
MARKET_COLUMNS = ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']

@lru_cache(maxsize=None)
def normalize_ticker(value):
    """Normalize a nonmissing ticker without removing legitimate dots or hyphens."""
    return re.sub(r'<[^>]*>', '', html.unescape(str(value))).strip().upper()

def prepare_frame(frame):
    """Keep source labels; parse dates and normalize tickers on an analysis copy."""
    result = frame.copy()
    if 'source_row' not in result:
        result.insert(0, 'source_row', np.arange(1, len(result) + 1))
    result['ticker_raw'] = result['ticker']
    result['ticker'] = result['ticker'].map(
        lambda value: pd.NA if pd.isna(value) else normalize_ticker(value)
    ).astype('string')
    result['ts'] = pd.to_datetime(result['ts'], format='%Y-%m-%d', errors='coerce')
    return result

data = prepare_frame(raw)
reference_data = prepare_frame(reference)
label_audit = (
    data.groupby(['ticker_raw', 'ticker'], dropna=False).size()
    .rename('row_count').reset_index()
)
print('Normalized stock count:', data['ticker'].nunique())
print('Tickers:', ', '.join(sorted(data['ticker'].dropna().unique())))
print('Rows with changed ticker labels:', int(data['ticker_raw'].ne(data['ticker']).sum()))
display(label_audit.loc[label_audit['ticker_raw'].ne(label_audit['ticker'])].head(8))
coverage = data.groupby('ticker').agg(
    first_date=('ts', 'min'), last_date=('ts', 'max'), rows=('source_row', 'size'),
    observed_dates=('ts', 'nunique')
)
display(coverage)

Normalized stock count: 30
Tickers: AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT
Rows with changed ticker labels: 39936


,ticker_raw,ticker,row_count
0,"<a target=""_blank""><span style=""color: blue"">AAPL</span></a>",AAPL,307
1,"<a target=""_blank""><span style=""color: blue"">AMGN</span></a>",AMGN,294
2,"<a target=""_blank""><span style=""color: blue"">AMZN</span></a>",AMZN,185
3,"<a target=""_blank""><span style=""color: blue"">AXP</span></a>",AXP,356
4,"<a target=""_blank""><span style=""color: blue"">BA</span></a>",BA,462
5,"<a target=""_blank""><span style=""color: blue"">CAT</span></a>",CAT,394
6,"<a target=""_blank""><span style=""color: blue"">CRM</span></a>",CRM,129
7,"<a target=""_blank""><span style=""color: blue"">CSCO</span></a>",CSCO,230


,first_date,last_date,rows,observed_dates
ticker,,,,
AAPL,1980-12-12,2024-10-18,11942,11050
AMGN,1983-06-17,2024-10-18,11290,10421
AMZN,1997-05-15,2024-12-06,7458,6937
AXP,1972-04-03,2024-10-18,14260,13252
BA,1962-01-02,2024-10-18,16980,15784
CAT,1962-01-02,2024-10-18,17044,15808
CRM,2004-06-23,2024-10-18,5525,5117
CSCO,1990-02-16,2024-10-18,9427,8734
CVX,1962-01-02,2024-12-06,17073,15843


## 1.2 Detect quality issues

The report permits several findings for the same source row. Counts below are **findings**, not a count of
unique defective observations.

- **High:** missing/invalid prices, invalid identifiers/dates, conflicting keys or impossible OHLC bounds.
- **Medium:** redundant records, missing sessions/volume, or extreme returns requiring investigation.
- **Low:** ticker presentation changes and zero-volume days, which can be legitimate.

Missing adjusted close is a high-severity validation failure for later risk reporting, even when a defensible repair may exist.
An outlier is a review flag, not proof of an error or a reason to delete a row.

In [4]:
REPORT_COLUMNS = ['ticker', 'issue_type', 'issue_date', 'description', 'severity', 'source_row']

def report_rows(frame, mask, issue_type, description, severity):
    """Return one finding per selected observation; preserve the original row identifier."""
    selected = frame.loc[mask, ['ticker', 'ts', 'source_row']].copy()
    selected = selected.rename(columns={'ts': 'issue_date'})
    selected['issue_type'] = issue_type
    selected['description'] = description
    selected['severity'] = severity
    return selected.reindex(columns=REPORT_COLUMNS)

def detect_missing_values(frame):
    """Report missing cells in the eight supplied market columns."""
    reports = []
    for column in MARKET_COLUMNS:
        reports.append(report_rows(
            frame, frame[column].isna(), f'missing_{column}',
            f'{column} is missing (or could not be parsed for ts).',
            'medium' if column == 'volume' else 'high'
        ))
    return pd.concat(reports, ignore_index=True)

def detect_ticker_formatting(frame):
    """Audit presentation changes separately from invalid or absent identifiers."""
    changed = frame['ticker_raw'].notna() & frame['ticker_raw'].ne(frame['ticker'])
    invalid = frame['ticker'].notna() & ~frame['ticker'].str.fullmatch(r'[A-Z0-9][A-Z0-9.\-]*', na=False)
    return pd.concat([
        report_rows(frame, changed, 'ticker_formatting',
                    'Raw label differs from normalized ticker; see label_audit.', 'low'),
        report_rows(frame, invalid, 'invalid_ticker',
                    'Ticker is empty or has unexpected characters after normalization.', 'high')
    ], ignore_index=True)

def duplicate_masks(frame):
    """Distinguish identical market records from distinct records sharing a valid key."""
    exact_duplicate = frame.duplicated(subset=MARKET_COLUMNS, keep='first')
    distinct = frame.loc[~exact_duplicate]
    valid_key = distinct['ticker'].notna() & distinct['ts'].notna()
    conflict_keys = distinct.loc[valid_key & distinct.duplicated(['ticker', 'ts'], keep=False),
                                 ['ticker', 'ts']].drop_duplicates()
    keys = pd.MultiIndex.from_frame(frame[['ticker', 'ts']])
    conflicting = pd.Series(keys.isin(pd.MultiIndex.from_frame(conflict_keys)), index=frame.index)
    return exact_duplicate, conflicting

def detect_duplicates(frame):
    """Mark redundant copies and every observation belonging to a conflicting key."""
    exact, conflict = duplicate_masks(frame)
    return pd.concat([
        report_rows(frame, exact, 'exact_duplicate',
                    'An identical normalized market record appeared earlier.', 'medium'),
        report_rows(frame, conflict, 'conflicting_key',
                    'Distinct market records share this normalized ticker/date; review all versions.', 'high')
    ], ignore_index=True)

def detect_price_and_volume_issues(frame):
    """Check price validity, raw OHLC bounds, and missing/invalid/zero volume."""
    reports = []
    for column in PRICE_COLUMNS:
        values = pd.to_numeric(frame[column], errors='coerce')
        invalid = frame[column].notna() & (~np.isfinite(values) | values.le(0))
        reports.append(report_rows(frame, invalid, f'invalid_{column}',
                                   f'{column} is nonpositive, nonnumeric or non-finite.', 'high'))
    raw_prices = frame[RAW_PRICE_COLUMNS].apply(pd.to_numeric, errors='coerce')
    finite = pd.DataFrame(np.isfinite(raw_prices), index=frame.index, columns=RAW_PRICE_COLUMNS)
    comparisons = [
        ('high_below_low', 'high', 'low', 'high is below low.'),
        ('open_below_low', 'open', 'low', 'open is below low.'),
        ('open_above_high', 'high', 'open', 'open is above high.'),
        ('close_below_low', 'close', 'low', 'close is below low.'),
        ('close_above_high', 'high', 'close', 'close is above high.')
    ]
    for issue, left, right, description in comparisons:
        mask = finite[left] & finite[right] & raw_prices[left].lt(raw_prices[right])
        reports.append(report_rows(frame, mask, issue, description, 'high'))
    reports.append(report_rows(frame, frame[RAW_PRICE_COLUMNS].isna().all(axis=1),
                               'all_raw_prices_missing', 'All four raw OHLC prices are missing.', 'high'))
    volume = pd.to_numeric(frame['volume'], errors='coerce')
    invalid_volume = frame['volume'].notna() & (~np.isfinite(volume) | volume.lt(0))
    reports.append(report_rows(frame, invalid_volume, 'invalid_volume',
                               'Volume is negative, nonnumeric or non-finite.', 'high'))
    reports.append(report_rows(frame, volume.eq(0), 'zero_volume',
                               'Reported volume is zero; review for possible stale data.', 'low'))
    return pd.concat(reports, ignore_index=True)

def detect_missing_sessions(frame, calendar):
    """Compare observed dates with the supplied reference history for each ticker."""
    reports, summaries = [], []
    actual = frame.loc[frame['ticker'].notna() & frame['ts'].notna()]
    expected = calendar.loc[calendar['ticker'].notna() & calendar['ts'].notna()]
    for ticker, expected_rows in expected.groupby('ticker', sort=True):
        expected_dates = pd.DatetimeIndex(expected_rows['ts'].drop_duplicates()).sort_values()
        observed_dates = pd.DatetimeIndex(actual.loc[actual['ticker'].eq(ticker), 'ts'].unique())
        missing = expected_dates.difference(observed_dates)
        extra = observed_dates.difference(expected_dates)
        start, end = expected_dates.min(), expected_dates.max()
        sentence = (f'Ticker {ticker} is missing {len(missing)} trading days '
                    f'between {start:%Y-%m-%d} and {end:%Y-%m-%d}.')
        summaries.append({'ticker': ticker, 'reference_start': start, 'reference_end': end,
                          'expected_sessions': len(expected_dates), 'missing_sessions': len(missing),
                          'dates_outside_reference': len(extra), 'summary': sentence})
        for date in missing:
            reports.append({'ticker': ticker, 'issue_type': 'missing_session', 'issue_date': date,
                            'description': 'Date exists in the supplied ohlc reference for this ticker, but no dirty row exists.',
                            'severity': 'medium', 'source_row': pd.NA})
    return pd.DataFrame(reports, columns=REPORT_COLUMNS), pd.DataFrame(summaries)

def calculate_diagnostic_returns(frame):
    """Use sorted, unambiguous keys and adjacent positive finite raw closes; never fill gaps."""
    exact, conflict = duplicate_masks(frame)
    # Temporary diagnostic view only: raw and the database remain unchanged.
    usable = ~exact & ~conflict & frame['ticker'].notna() & frame['ts'].notna()
    result = frame.loc[usable].sort_values(['ticker', 'ts']).copy()
    close = pd.to_numeric(result['close'], errors='coerce')
    previous = close.groupby(result['ticker']).shift(1)
    valid_pair = np.isfinite(close) & close.gt(0) & np.isfinite(previous) & previous.gt(0)
    result['daily_return'] = ((close - previous) / previous).where(valid_pair)
    result['mean_return'] = result.groupby('ticker')['daily_return'].transform('mean')
    result['std_return'] = result.groupby('ticker')['daily_return'].transform(lambda x: x.std(ddof=1))
    std = result['std_return']
    eligible = np.isfinite(result['daily_return']) & np.isfinite(std) & std.gt(0)
    result['return_outlier'] = eligible & (
        (result['daily_return'] - result['mean_return']).abs() > 3 * std
    )
    return result

def detect_return_outliers(return_data):
    """Report both tails of the retrospective, per-ticker sample-standard-deviation screen."""
    selected = return_data.loc[return_data['return_outlier']].copy()
    report = report_rows(selected, pd.Series(True, index=selected.index),
                         'return_outlier', '', 'medium')
    report['description'] = selected.apply(
        lambda row: f'Raw close return {row.daily_return:.2%}; ticker mean {row.mean_return:.2%}, '
                    f'sample std {row.std_return:.2%}. Review, not automatic removal.', axis=1
    )
    return report

def build_quality_report(frame, calendar):
    """Return the complete diagnosis, trading-calendar summary and diagnostic return view."""
    missing_sessions, session_summary = detect_missing_sessions(frame, calendar)
    returns = calculate_diagnostic_returns(frame)
    report = pd.concat([
        detect_missing_values(frame), detect_ticker_formatting(frame), detect_duplicates(frame),
        detect_price_and_volume_issues(frame), missing_sessions, detect_return_outliers(returns)
    ], ignore_index=True)
    report = report.sort_values(['ticker', 'issue_date', 'issue_type', 'source_row'], na_position='last')
    return report.reset_index(drop=True), session_summary, returns

### Missing cells and trading-session coverage
The calendar is the set of dates in the supplied `ohlc` table **for each stock**, over that stock's own
supplied history. It is not a weekday-only calendar and is not independently verified against an exchange.
A row with a missing price still counts as an observed date. No absent rows or prices are invented.

In [5]:
quality_report, session_summary, diagnostic_returns = build_quality_report(data, reference_data)
missing_counts = raw[MARKET_COLUMNS].isna().sum().rename('missing_cells').to_frame()
display(missing_counts)
print('Unparseable/missing dates:', int(data['ts'].isna().sum()))
print('Reference duplicate ticker/date keys:', int(reference_data.duplicated(['ticker', 'ts']).sum()))
print('Dirty tickers absent from reference:', sorted(set(data['ticker'].dropna()) - set(reference_data['ticker'].dropna())))
for sentence in session_summary['summary']:
    print(sentence)
display(session_summary.drop(columns='summary'))

,missing_cells
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


Unparseable/missing dates: 0
Reference duplicate ticker/date keys: 0


Dirty tickers absent from reference: []
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18.
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18.
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06.
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18.
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18.
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18.
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06.
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06.
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18.
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18.
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker IBM is missing 0

,ticker,reference_start,reference_end,expected_sessions,missing_sessions,dates_outside_reference
0,AAPL,1980-12-12,2024-10-18,11050,0,0
1,AMGN,1983-06-17,2024-10-18,10421,0,0
2,AMZN,1997-05-15,2024-12-06,6937,0,0
3,AXP,1972-04-03,2024-10-18,13252,0,0
4,BA,1962-01-02,2024-10-18,15784,0,0
5,CAT,1962-01-02,2024-10-18,15808,0,0
6,CRM,2004-06-23,2024-10-18,5117,0,0
7,CSCO,1990-02-16,2024-10-18,8734,0,0
8,CVX,1962-01-02,2024-12-06,15843,0,0
9,DIS,1962-01-02,2024-12-06,15867,0,0


### Data Quality Report
The full report is kept in memory, with a `source_row` audit field in addition to the five required columns.
Identical records are identified **after** ticker normalization. A separate temporary return-analysis view
counts one copy of each identical record and excludes every conflicting key; it never chooses an arbitrary version.
Every source row remains available in `data` and `raw`.

In [6]:
issue_counts = quality_report.groupby(['issue_type', 'severity']).size().rename('findings').reset_index()
display(issue_counts.sort_values('findings', ascending=False))
print('Total findings:', f'{len(quality_report):,}')
print('Distinct source rows with a finding:', f'{quality_report.source_row.nunique():,}')
display(quality_report.head(25))
# Show examples of substantive problems separately from ticker formatting.
substantive = quality_report.loc[quality_report['issue_type'].ne('ticker_formatting')]
display(substantive.groupby('issue_type', sort=True).head(2))
exact_mask, conflict_mask = duplicate_masks(data)
print('Redundant exact copies:', int(exact_mask.sum()))
print('Rows belonging to conflicting keys:', int(conflict_mask.sum()))
print('Distinct conflicting keys:', len(data.loc[conflict_mask, ['ticker', 'ts']].drop_duplicates()))
print('All observed nonpositive opening prices:')
display(data.loc[data['open'].le(0), ['source_row', 'ticker', 'ts', 'open', 'high', 'low', 'close']])

,issue_type,severity,findings
5,ticker_formatting,low,39936
0,exact_duplicate,medium,29460
2,missing_adj_close,high,19855
4,return_outlier,medium,5056
6,zero_volume,low,81
1,invalid_open,high,5
3,open_below_low,high,5


Total findings: 94,398
Distinct source rows with a finding: 87,488


,ticker,issue_type,issue_date,description,severity,source_row
0,AAPL,ticker_formatting,1980-12-19 00:00:00,Raw label differs from normalized ticker; see label_audit.,low,197771
1,AAPL,return_outlier,1980-12-26 00:00:00,"Raw close return 9.23%; ticker mean 0.11%, sample std 2.78%. Review, not automatic removal.",medium,393433
2,AAPL,ticker_formatting,1980-12-29 00:00:00,Raw label differs from normalized ticker; see label_audit.,low,137992
3,AAPL,exact_duplicate,1981-01-02 00:00:00,An identical normalized market record appeared earlier.,medium,218477
4,AAPL,exact_duplicate,1981-01-05 00:00:00,An identical normalized market record appeared earlier.,medium,343475
5,AAPL,missing_adj_close,1981-01-08 00:00:00,adj_close is missing (or could not be parsed for ts).,high,241024
6,AAPL,missing_adj_close,1981-01-13 00:00:00,adj_close is missing (or could not be parsed for ts).,high,229550
7,AAPL,ticker_formatting,1981-02-02 00:00:00,Raw label differs from normalized ticker; see label_audit.,low,269965
8,AAPL,ticker_formatting,1981-02-04 00:00:00,Raw label differs from normalized ticker; see label_audit.,low,218834
9,AAPL,exact_duplicate,1981-02-10 00:00:00,An identical normalized market record appeared earlier.,medium,251637


,ticker,issue_type,issue_date,description,severity,source_row
1,AAPL,return_outlier,1980-12-26 00:00:00,"Raw close return 9.23%; ticker mean 0.11%, sample std 2.78%. Review, not automatic removal.",medium,393433
3,AAPL,exact_duplicate,1981-01-02 00:00:00,An identical normalized market record appeared earlier.,medium,218477
4,AAPL,exact_duplicate,1981-01-05 00:00:00,An identical normalized market record appeared earlier.,medium,343475
5,AAPL,missing_adj_close,1981-01-08 00:00:00,adj_close is missing (or could not be parsed for ts).,high,241024
6,AAPL,missing_adj_close,1981-01-13 00:00:00,adj_close is missing (or could not be parsed for ts).,high,229550
22,AAPL,return_outlier,1981-04-02 00:00:00,"Raw close return 8.77%; ticker mean 0.11%, sample std 2.78%. Review, not automatic removal.",medium,139447
7355,AXP,zero_volume,1972-12-14 00:00:00,Reported volume is zero; review for possible stale data.,low,243678
7356,AXP,zero_volume,1972-12-14 00:00:00,Reported volume is zero; review for possible stale data.,low,370681
22217,CVX,invalid_open,1962-01-02 00:00:00,"open is nonpositive, nonnumeric or non-finite.",high,287344
22218,CVX,open_below_low,1962-01-02 00:00:00,open is below low.,high,287344


Redundant exact copies: 29460
Rows belonging to conflicting keys: 0
Distinct conflicting keys: 0
All observed nonpositive opening prices:


,source_row,ticker,ts,open,high,low,close
99637,99638,PG,1962-01-02,0.0,1.44000,1.42000,1.43000
224790,224791,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955
287343,287344,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000
350808,350809,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338
364568,364569,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537


### Return outliers
Use simple **raw-close** returns here, as requested. These differ from the adjusted-close log returns
required later for risk reports. The mean and sample standard deviation (`ddof=1`) use all defined
returns for each ticker. The threshold is retrospective and flags both tails. Zero or undefined
standard deviation produces no outlier flags. Returns require two adjacent valid observations;
no missing close is silently forward-filled.

A large raw return may reflect a corporate action or a data error. This screen alone cannot establish the cause.

In [7]:
return_overview = diagnostic_returns.groupby('ticker').agg(
    defined_returns=('daily_return', 'count'), mean_return=('daily_return', 'mean'),
    sample_std=('daily_return', lambda values: values.std(ddof=1)), outliers=('return_outlier', 'sum')
)
display(return_overview)
outliers = diagnostic_returns.loc[diagnostic_returns['return_outlier']].copy()
print('Negative-return flags:', int(outliers['daily_return'].lt(0).sum()))
print('Positive-return flags:', int(outliers['daily_return'].gt(0).sum()))
display(outliers.assign(absolute_return=outliers.daily_return.abs()).nlargest(15, 'absolute_return')[
    ['source_row', 'ticker', 'ts', 'close', 'daily_return', 'mean_return', 'std_return']
])

,defined_returns,mean_return,sample_std,outliers
ticker,,,,
AAPL,11049,0.001074,0.027846,143
AMGN,10420,0.000937,0.023731,172
AMZN,6936,0.001729,0.035298,147
AXP,13251,0.000577,0.022122,199
BA,15783,0.000561,0.021421,210
CAT,15807,0.000521,0.018588,221
CRM,5116,0.001174,0.026448,82
CSCO,8733,0.001064,0.024872,148
CVX,15842,0.000374,0.016185,178


Negative-return flags: 2226
Positive-return flags: 2830


,source_row,ticker,ts,close,daily_return,mean_return,std_return
384157,384158,TRV,1976-10-28,9.220000,1.000000,0.000573,0.024232
146175,146176,TRV,1981-08-10,12.060000,1.000000,0.000573,0.024232
8975,8976,TRV,1978-12-20,8.030000,0.997512,0.000573,0.024232
379437,379438,MCD,1968-01-02,0.174900,-0.753620,0.000678,0.018524
110212,110213,AAPL,2000-09-29,0.459820,-0.518695,0.001074,0.027846
148374,148375,TRV,1981-08-11,6.030000,-0.500000,0.000573,0.024232
273517,273518,TRV,1976-10-29,4.690000,-0.491323,0.000573,0.024232
273555,273556,TRV,1978-12-21,4.090000,-0.490660,0.000573,0.024232
244847,244848,NVDA,2000-03-07,0.173570,0.424106,0.001966,0.037879
386553,386554,NVDA,2004-08-06,0.078584,-0.352314,0.001966,0.037879


### Small controlled examples
These examples are separate from the real data. They demonstrate detection of missing cells, a missing
session, both outlier tails, duplicates/conflicts, invalid prices, impossible OHLC ranges and volume issues,
even if a particular fault is absent from the supplied database. They do not repair data.
Repair demonstrations belong to the later cleaning stage.

In [8]:
def example_frame(periods=4):
    """Create a tiny valid daily-price fixture; its dates define its own reference calendar."""
    return pd.DataFrame({
        'ts': pd.date_range('2024-01-02', periods=periods, freq='B').strftime('%Y-%m-%d'),
        'open': 100.0, 'high': 102.0, 'low': 98.0, 'close': 100.0,
        'adj_close': 100.0, 'volume': 1000.0, 'ticker': 'DEMO'
    })

fixture = example_frame()
injected = fixture.copy()
injected.loc[0, 'open'] = np.nan
injected.loc[0, 'volume'] = np.nan
injected.loc[1, 'high'] = 95.0
injected.loc[1, 'volume'] = -1.0
injected.loc[2, 'close'] = 0.0
injected.loc[2, 'adj_close'] = np.inf
injected.loc[2, 'volume'] = 0.0
injected.loc[3, 'ticker'] = '<b> demo </b>'
injected = pd.concat([injected, injected.iloc[[3]]], ignore_index=True)
conflict = injected.iloc[[3]].copy()
conflict['volume'] = 500.0
injected = pd.concat([injected, conflict], ignore_index=True)
injected_report, _, _ = build_quality_report(prepare_frame(injected), prepare_frame(fixture))
expected_issues = {'missing_open', 'missing_volume', 'high_below_low', 'open_above_high',
                   'close_above_high', 'invalid_close', 'invalid_adj_close', 'invalid_volume',
                   'zero_volume', 'ticker_formatting', 'exact_duplicate', 'conflicting_key'}
assert expected_issues.issubset(set(injected_report['issue_type']))
display(injected_report)

removed_date = fixture.loc[1, 'ts']
gap_report, gap_summary = detect_missing_sessions(
    prepare_frame(fixture.drop(index=1)), prepare_frame(fixture)
)
assert len(gap_report) == 1 and gap_report.iloc[0]['issue_date'] == pd.Timestamp(removed_date)
print(gap_summary.iloc[0]['summary'])
display(gap_report)

# Two opposite 50% moves among otherwise zero returns must both be detected.
tail_fixture = example_frame(50)
synthetic_returns = np.zeros(49)
synthetic_returns[10], synthetic_returns[30] = 0.5, -0.5
tail_fixture['close'] = np.r_[100.0, 100.0 * np.cumprod(1 + synthetic_returns)]
tail_diagnosis = calculate_diagnostic_returns(prepare_frame(tail_fixture))
flagged_returns = tail_diagnosis.loc[tail_diagnosis.return_outlier, 'daily_return']
assert len(flagged_returns) == 2 and (flagged_returns.lt(0).any() and flagged_returns.gt(0).any())
constant_diagnosis = calculate_diagnostic_returns(prepare_frame(example_frame()))
assert constant_diagnosis.std_return.eq(0).all() and not constant_diagnosis.return_outlier.any()
single_diagnosis = calculate_diagnostic_returns(prepare_frame(example_frame(1)))
assert single_diagnosis.std_return.isna().all() and not single_diagnosis.return_outlier.any()
assert normalize_ticker(' <b>brk.b</b> ') == 'BRK.B'
print('Controlled detection checks passed, including both tails and zero/undefined standard deviation.')

,ticker,issue_type,issue_date,description,severity,source_row
0,DEMO,missing_open,2024-01-02 00:00:00,open is missing (or could not be parsed for ts).,high,1
1,DEMO,missing_volume,2024-01-02 00:00:00,volume is missing (or could not be parsed for ts).,medium,1
2,DEMO,close_above_high,2024-01-03 00:00:00,close is above high.,high,2
3,DEMO,high_below_low,2024-01-03 00:00:00,high is below low.,high,2
4,DEMO,invalid_volume,2024-01-03 00:00:00,"Volume is negative, nonnumeric or non-finite.",high,2
5,DEMO,open_above_high,2024-01-03 00:00:00,open is above high.,high,2
6,DEMO,close_below_low,2024-01-04 00:00:00,close is below low.,high,3
7,DEMO,invalid_adj_close,2024-01-04 00:00:00,"adj_close is nonpositive, nonnumeric or non-finite.",high,3
8,DEMO,invalid_close,2024-01-04 00:00:00,"close is nonpositive, nonnumeric or non-finite.",high,3
9,DEMO,zero_volume,2024-01-04 00:00:00,Reported volume is zero; review for possible stale data.,low,3


Ticker DEMO is missing 1 trading days between 2024-01-02 and 2024-01-05.


,ticker,issue_type,issue_date,description,severity,source_row
0,DEMO,missing_session,2024-01-03,"Date exists in the supplied ohlc reference for this ticker, but no dirty row exists.",medium,<NA>


Controlled detection checks passed, including both tails and zero/undefined standard deviation.


## Initial findings before cleaning

In this supplied snapshot, normalization identifies 30 actual stocks rather than 150 presentation labels.
There are 29,460 redundant copies, 19,855 missing adjusted-close cells, and five zero opening prices
that also violate the daily low. There are no conflicting ticker/date keys and no absent sessions
relative to the supplied reference histories. The 81 zero-volume observations require review;
zero volume alone is not proof of a bad record. These source-row counts can include redundant copies.

The 5,056 flagged returns are calculated after redundant copies are excluded from the diagnostic view.
They are not 5,056 established errors. Both positive and negative extreme returns occur, and the
largest moves need investigation before choosing repairs. Different tickers also have different final dates:
October 18 or December 6, 2024. Later cross-stock reports must disclose actual coverage.

All five zero opening prices occur at the start of their respective supplied histories. There is no previous
day available for a defensible forward fill; later cleaning should either justify another repair or quarantine
them with a reason. Missing adjusted closes need a consistent date-specific adjustment convention,
not an unverified carry-forward across possible corporate actions.

At this point no repairs have been performed. The next cell verifies that inspection left the database
unchanged. Section 1.3 below extends this diagnosis with reference-adjustment checks and then cleans the data.

In [9]:
for row in issue_counts.sort_values('issue_type').itertuples(index=False):
    print(f'{row.issue_type}: {row.findings:,} findings ({row.severity})')
print(f'Missing reference sessions: {session_summary.missing_sessions.sum():,}')
print(f'Dates outside the per-ticker reference: {session_summary.dates_outside_reference.sum():,}')
assert file_sha256(DB_PATH) == original_database_hash, 'The working database changed during inspection.'
assert len(raw) == len(data), 'The inspection must retain every source row.'
print('Database SHA256 unchanged:', original_database_hash)
print('Tables at inspection start:', ', '.join(original_tables))
print('Read-only inspection complete; cleaning begins in section 1.3 below.')

exact_duplicate: 29,460 findings (medium)
invalid_open: 5 findings (high)
missing_adj_close: 19,855 findings (high)
open_below_low: 5 findings (high)
return_outlier: 5,056 findings (medium)
ticker_formatting: 39,936 findings (low)
zero_volume: 81 findings (low)
Missing reference sessions: 0
Dates outside the per-ticker reference: 0


Database SHA256 unchanged: 7b44232f0717c3500c2fd5d0133d4f481ba0929ceaf3c3cf92ae50b015aeea73
Tables at inspection start: market_caps, ohlc, ohlc_hw, ohlc_hw_cleaned
Read-only inspection complete; cleaning begins in section 1.3 below.


## 1.3 Clean and validate

### Additional investigation: adjusted-price corruption
Positive and finite does not guarantee a plausible adjusted price. Match the supplied `ohlc` reference
by normalized ticker and exact date. Use its **date-specific** adjustment factor
`reference adj_close / reference close`, never a factor carried across dates.
Compare the observed adjusted close with `observed close × reference factor`.
An isolated mismatch is investigated separately from the raw-close return screen.
The reference is the assignment's supplied adjustment convention, not independent proof of market truth.

The raw OHLC values and volumes are not replaced wholesale with reference values.

In [10]:
def reference_adjustment_data(frame, calendar):
    """Match only valid, unique reference keys; expose the adjustment basis, not replacement OHLC."""
    ref = calendar.loc[calendar['ticker'].notna() & calendar['ts'].notna(),
                       ['ticker', 'ts', 'close', 'adj_close']].copy()
    if ref.duplicated(['ticker', 'ts']).any():
        raise ValueError('Reference ticker/date keys must be unique.')
    ref = ref.rename(columns={'close': 'reference_close', 'adj_close': 'reference_adj_close'})
    matched = frame[['source_row', 'ticker', 'ts', 'close', 'adj_close']].merge(
        ref, on=['ticker', 'ts'], how='left', validate='many_to_one'
    ).set_index('source_row')
    valid = (np.isfinite(matched['reference_close']) & matched['reference_close'].gt(0)
             & np.isfinite(matched['reference_adj_close']) & matched['reference_adj_close'].gt(0))
    matched['reference_adjustment_factor'] = (
        matched['reference_adj_close'] / matched['reference_close']
    ).where(valid)
    matched['expected_adj_close'] = matched['close'] * matched['reference_adjustment_factor']
    comparable = valid & np.isfinite(matched['adj_close']) & matched['adj_close'].gt(0)
    matched['adjustment_mismatch'] = comparable & ~np.isclose(
        matched['adj_close'], matched['expected_adj_close'], rtol=1e-8, atol=1e-10
    )
    return matched

adjustment_check = reference_adjustment_data(data, reference_data)
mismatch_ids = adjustment_check.index[adjustment_check['adjustment_mismatch']]
adjustment_issues = report_rows(
    data, data['source_row'].isin(mismatch_ids), 'adjustment_factor_mismatch',
    'Adjusted close disagrees with the matched date-specific ohlc adjustment convention.', 'high'
)
quality_report = pd.concat([quality_report, adjustment_issues], ignore_index=True).sort_values(
    ['ticker', 'issue_date', 'issue_type', 'source_row'], na_position='last'
).reset_index(drop=True)
issue_counts = quality_report.groupby(['issue_type', 'severity']).size().rename('findings').reset_index()
unique_adjustments = adjustment_check.loc[
    ~adjustment_check[['ticker', 'ts', 'close', 'adj_close']].duplicated()
]
mismatches = unique_adjustments.loc[unique_adjustments['adjustment_mismatch']].copy()
mismatches['observed_to_expected'] = mismatches['adj_close'] / mismatches['expected_adj_close']
print('Source-row adjustment mismatches (including repeated records):', len(adjustment_issues))
print('Distinct-record adjustment mismatches:', len(mismatches))
print('Observed/reference ratio range:', mismatches['observed_to_expected'].min(),
      'to', mismatches['observed_to_expected'].max())
display(mismatches.nlargest(8, 'observed_to_expected')[
    ['ticker', 'ts', 'close', 'adj_close', 'expected_adj_close', 'observed_to_expected']
])
# Diagnose all flagged raw returns, not just the largest examples.
reference_returns = calculate_diagnostic_returns(reference_data).set_index(['ticker', 'ts'])
return_investigation = diagnostic_returns.loc[diagnostic_returns['return_outlier'],
    ['source_row', 'ticker', 'ts', 'close', 'daily_return']].merge(
    reference_returns[['close', 'daily_return', 'adj_close']].rename(columns={
        'close': 'reference_close', 'daily_return': 'reference_raw_return',
        'adj_close': 'reference_adj_close'
    }), on=['ticker', 'ts'], how='left', validate='one_to_one'
)
return_investigation['close_matches_reference'] = np.isclose(
    return_investigation['close'], return_investigation['reference_close'], rtol=1e-8, atol=1e-10
)
return_investigation['review_conclusion'] = np.where(
    return_investigation['close_matches_reference'],
    'Matches supplied reference; economic cause unverified; retain with review flag.',
    'Does not match supplied reference; additional investigation required.'
)
print('Extreme raw returns reviewed:', len(return_investigation))
print('Matching reference closes:', int(return_investigation.close_matches_reference.sum()))
display(return_investigation.assign(size=return_investigation.daily_return.abs()).nlargest(10, 'size'))

Source-row adjustment mismatches (including repeated records): 11265
Distinct-record adjustment mismatches: 10417
Observed/reference ratio range: 4.002156378267482 to 23.9984386284633


,ticker,ts,close,adj_close,expected_adj_close,observed_to_expected
source_row,,,,,,
395813,NKE,2008-06-24,16.49000,319.899187,13.33000,23.998439
208807,TRV,1984-09-26,6.47000,47.749865,1.99000,23.994907
367076,NKE,2007-02-01,12.58000,238.987226,9.96000,23.994701
129334,AXP,1999-08-27,42.59000,726.546776,30.28000,23.994279
377690,MRK,1989-07-24,11.78000,87.088464,3.63000,23.991312
391744,VZ,2008-05-19,36.53000,377.838300,15.75000,23.989733
158017,JPM,1990-09-17,6.50000,55.166880,2.30000,23.985600
152153,NVDA,2003-07-23,0.17975,3.953666,0.16487,23.980503


Extreme raw returns reviewed: 5056
Matching reference closes: 5056


,source_row,ticker,ts,close,daily_return,reference_close,reference_raw_return,reference_adj_close,close_matches_reference,review_conclusion,size
4425,384158,TRV,1976-10-28,9.220000,1.000000,9.220000,1.000000,3.210000,True,Matches supplied reference; economic cause unverified; retain with review flag.,1.000000
4429,146176,TRV,1981-08-10,12.060000,1.000000,12.060000,1.000000,4.200000,True,Matches supplied reference; economic cause unverified; retain with review flag.,1.000000
4427,8976,TRV,1978-12-20,8.030000,0.997512,8.030000,0.997512,2.800000,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.997512
3066,379438,MCD,1968-01-02,0.174900,-0.753620,0.174900,-0.753620,0.075921,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.753620
98,110213,AAPL,2000-09-29,0.459820,-0.518695,0.459820,-0.518695,0.388310,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.518695
4430,148375,TRV,1981-08-11,6.030000,-0.500000,6.030000,-0.500000,1.570000,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.500000
4426,273518,TRV,1976-10-29,4.690000,-0.491323,4.690000,-0.491323,0.997070,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.491323
4428,273556,TRV,1978-12-21,4.090000,-0.490660,4.090000,-0.490660,0.924090,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.490660
3954,244848,NVDA,2000-03-07,0.173570,0.424106,0.173570,0.424106,0.159200,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.424106
3996,386554,NVDA,2004-08-06,0.078584,-0.352314,0.078584,-0.352314,0.072076,True,Matches supplied reference; economic cause unverified; retain with review flag.,0.352314


### Cleaning policy

1. Normalize tickers and keep original labels. Remove identical market records after normalization.
   Quarantine distinct conflicting ticker/date records rather than selecting a convenient version.
2. Remove rows with all four raw prices missing **before** filling. Work in ticker/date order.
   Treat nonpositive/non-finite raw prices as missing; treat an out-of-range open or close as suspect.
   Forward-fill within each ticker and recheck every raw bound. Ambiguous inverted high/low ranges
   and unresolved leading gaps are quarantined, not extrapolated. A repaired open/close still has to
   lie within that day's valid low/high range; forward filling is not automatically accepted.
3. Missing volume becomes zero with an explicit `volume_missing_filled` flag. Negative or infinite
   volume is quarantined. An original zero volume is retained with a separate review flag.
4. Reconstruct missing/invalid adjusted close and the investigated reference-adjustment mismatches
   as validated raw close times the exact-date reference factor. Recompute adjusted close after a
   raw-close repair too. Preserve original values, factor and reason. If the needed factor is unavailable,
   quarantine the unresolved row. Unchanged original vendor values without a reference factor are
   not claimed to have been independently verified.
5. The raw-close outliers match the supplied reference. Keep them with a review flag and an
   explicit unverified-cause note. No split, merger or news explanation is asserted.
   An optional evidence table supports interpolation of **confirmed** close errors between valid
   neighbors only; no real observation is classified as a confirmed error on this evidence alone.

The five observed zero opens occur at the start of their supplied histories and will be quarantined.
The reference contains these same zero opens, so it cannot justify filling them in.

In [11]:
def raw_bounds_valid(frame):
    """Check finite positive raw prices and both open/close within the daily low/high."""
    prices = frame[RAW_PRICE_COLUMNS].apply(pd.to_numeric, errors='coerce')
    valid = pd.Series(np.isfinite(prices).all(axis=1), index=frame.index) & prices.gt(0).all(axis=1)
    return (valid & prices['low'].le(prices['high'])
            & prices['open'].between(prices['low'], prices['high'])
            & prices['close'].between(prices['low'], prices['high']))

def interpolate_confirmed_close_errors(frame, evidence):
    """Interpolate only documented error keys inside a sorted ticker, without endpoint extrapolation."""
    result = frame.copy()
    result['confirmed_close_error'] = False
    result['close_error_interpolated'] = False
    result['close_error_evidence'] = ''
    if evidence is None or evidence.empty:
        return result
    required = ['ticker', 'ts', 'evidence']
    if not set(required).issubset(evidence.columns):
        raise ValueError('Confirmed errors require ticker, ts and evidence.')
    errors = evidence[required].copy()
    errors['ticker'] = errors['ticker'].map(normalize_ticker)
    errors['ts'] = pd.to_datetime(errors['ts'], format='%Y-%m-%d', errors='raise')
    if errors.duplicated(['ticker', 'ts']).any() or errors['evidence'].fillna('').str.strip().eq('').any():
        raise ValueError('Each confirmed error must have a unique key and nonempty evidence.')
    evidence_by_key = errors.set_index(['ticker', 'ts'])['evidence']
    reasons = pd.Series(pd.MultiIndex.from_frame(result[['ticker', 'ts']]).map(evidence_by_key),
                        index=result.index)
    target = reasons.notna()
    result.loc[target, 'confirmed_close_error'] = True
    result.loc[target, 'close_error_evidence'] = reasons.loc[target]
    original_missing = result['close'].isna()
    result.loc[target, 'close'] = np.nan
    candidates = result.groupby('ticker', sort=False)['close'].transform(
        lambda values: values.interpolate(method='linear', limit_area='inside')
    )
    # A genuine missing observation remains for the specified forward-fill policy.
    repaired = target & candidates.notna()
    result.loc[repaired, 'close'] = candidates.loc[repaired]
    result.loc[repaired, 'close_error_interpolated'] = True
    result.loc[original_missing & ~target, 'close'] = np.nan
    return result

def clean_equity_data(source, calendar, confirmed_close_errors=None):
    """Clean a copy, preserve audit fields and partition every source row into one disposition."""
    work = prepare_frame(source).set_index('source_row', drop=False)
    if not work.index.is_unique:
        raise ValueError('source_row identifiers must be unique.')
    ledger = work[['source_row', 'ticker_raw', 'ticker', 'ts']].copy()
    ledger['disposition'] = 'pending'
    ledger['reason'] = ''
    rejected = []
    for column in PRICE_COLUMNS + ['volume']:
        work['original_' + column] = work[column]
        work[column] = pd.to_numeric(work[column], errors='coerce')
    work['ticker_normalized'] = work['ticker_raw'].ne(work['ticker']).fillna(False)

    def exclude(mask, disposition, reason):
        """Move selected rows out of work while retaining an auditable original-row partition."""
        nonlocal work
        mask = mask.fillna(False)
        ids = work.index[mask]
        if len(ids):
            ledger.loc[ids, 'disposition'] = disposition
            ledger.loc[ids, 'reason'] = reason
            rejected.append(work.loc[ids].assign(disposition=disposition, reason=reason))
            work = work.loc[~mask].copy()

    exact, _ = duplicate_masks(work)
    exclude(exact, 'removed', 'redundant_exact_duplicate')
    exclude(work[RAW_PRICE_COLUMNS].isna().all(axis=1), 'removed', 'all_raw_prices_missing_before_fill')
    invalid_key = work['ts'].isna() | ~work['ticker'].str.fullmatch(r'[A-Z0-9][A-Z0-9.\-]*', na=False)
    exclude(invalid_key, 'quarantined', 'invalid_ticker_or_date')
    _, conflict = duplicate_masks(work)
    exclude(conflict, 'quarantined', 'conflicting_ticker_date_records')
    work = work.sort_values(['ticker', 'ts'])
    work['return_outlier_before'] = work['source_row'].isin(
        calculate_diagnostic_returns(prepare_frame(source)).loc[lambda x: x.return_outlier, 'source_row']
    )
    matched = reference_adjustment_data(work, calendar)
    work['reference_adjustment_factor'] = matched['reference_adjustment_factor']
    work['adjustment_mismatch_before'] = matched['adjustment_mismatch']
    exclude(work['high'].lt(work['low']), 'quarantined', 'ambiguous_inverted_high_low')

    for column in RAW_PRICE_COLUMNS:
        invalid = ~np.isfinite(work[column]) | work[column].le(0)
        work[column + '_invalid_original'] = invalid & work[column].notna()
        work.loc[invalid, column] = np.nan
    for column in ['open', 'close']:
        suspect = (work['low'].notna() & work['high'].notna() & work[column].notna()
                   & ~work[column].between(work['low'], work['high']))
        work[column + '_outside_bounds_original'] = suspect
        work.loc[suspect, column] = np.nan

    work = interpolate_confirmed_close_errors(work, confirmed_close_errors)
    for column in RAW_PRICE_COLUMNS:
        missing = work[column].isna()
        filled = work.groupby('ticker', sort=False)[column].ffill()
        fill_allowed = missing
        if column == 'close':
            # A confirmed endpoint error may not fall back to forward-fill extrapolation.
            fill_allowed = missing & ~work['confirmed_close_error']
        repaired = fill_allowed & filled.notna()
        work.loc[repaired, column] = filled.loc[repaired]
        work[column + '_forward_filled'] = repaired
    work['raw_bounds_valid_after_repair'] = raw_bounds_valid(work)
    exclude(~work['raw_bounds_valid_after_repair'], 'quarantined', 'unresolved_raw_price_or_ohlc_bounds')

    work['volume_missing_filled'] = work['volume'].isna()
    work.loc[work['volume_missing_filled'], 'volume'] = 0
    exclude(~np.isfinite(work['volume']) | work['volume'].lt(0), 'quarantined', 'invalid_volume')
    work['zero_volume_review'] = work['volume'].eq(0)

    factor = work['reference_adjustment_factor']
    factor_valid = np.isfinite(factor) & factor.gt(0)
    adjusted_invalid = ~np.isfinite(work['adj_close']) | work['adj_close'].le(0)
    close_changed = ~np.isclose(work['close'], work['original_close'], rtol=1e-8, atol=1e-10, equal_nan=True)
    expected = work['close'] * factor
    inconsistent = factor_valid & ~np.isclose(work['adj_close'], expected, rtol=1e-8, atol=1e-10)
    needs_adjustment = adjusted_invalid | close_changed | inconsistent
    repaired = needs_adjustment & factor_valid
    work['adj_close_repaired'] = repaired
    work['adj_close_repair_reason'] = ''
    work.loc[repaired & adjusted_invalid, 'adj_close_repair_reason'] = 'missing_or_invalid_adjusted_close'
    work.loc[repaired & ~adjusted_invalid & inconsistent, 'adj_close_repair_reason'] = 'reference_factor_mismatch'
    work.loc[repaired & close_changed, 'adj_close_repair_reason'] = 'raw_close_repaired_reapply_factor'
    work.loc[repaired, 'adj_close'] = expected.loc[repaired]
    work['adjustment_factor_source'] = np.where(factor_valid, 'ohlc: exact ticker/date', 'original_vendor_unverified')
    work['adjustment_consistency_verified'] = factor_valid & np.isclose(
        work['adj_close'], work['close'] * factor, rtol=1e-8, atol=1e-10
    )
    exclude(needs_adjustment & ~factor_valid, 'quarantined', 'required_reference_adjustment_factor_unavailable')
    exclude(~np.isfinite(work['adj_close']) | work['adj_close'].le(0), 'quarantined', 'unresolved_adjusted_price')

    final_returns = calculate_diagnostic_returns(work)
    after_flags = final_returns.set_index('source_row')['return_outlier']
    work['return_outlier_after'] = work['source_row'].map(after_flags).fillna(False).astype(bool)
    work['return_outlier_review'] = work['return_outlier_before'] | work['return_outlier_after']
    work['return_review_note'] = np.where(
        work['return_outlier_review'], 'Retained for review; economic cause unverified.', ''
    )
    ledger.loc[work.index, 'disposition'] = 'retained'
    ledger.loc[work.index, 'reason'] = 'passed_final_validation'
    assert ledger['disposition'].ne('pending').all()
    rejected_frame = pd.concat(rejected, ignore_index=True) if rejected else pd.DataFrame()
    quarantined = rejected_frame.loc[rejected_frame['disposition'].eq('quarantined')].copy() if len(rejected_frame) else rejected_frame.copy()
    removed = rejected_frame.loc[rejected_frame['disposition'].eq('removed')].copy() if len(rejected_frame) else rejected_frame.copy()
    return {'cleaned': work.reset_index(drop=True), 'quarantined': quarantined,
            'removed': removed, 'ledger': ledger.reset_index(drop=True)}

def validate_cleaned(frame):
    """Reject invalid retained data, duplicate keys, bad volume and inconsistent known adjustment factors."""
    assert frame['source_row'].is_unique
    assert not frame.duplicated(['ticker', 'ts']).any()
    assert frame['ts'].notna().all()
    assert frame['ticker'].str.fullmatch(r'[A-Z0-9][A-Z0-9.\-]*', na=False).all()
    assert np.isfinite(frame[PRICE_COLUMNS]).all().all() and frame[PRICE_COLUMNS].gt(0).all().all()
    assert raw_bounds_valid(frame).all()
    assert np.isfinite(frame['volume']).all() and frame['volume'].ge(0).all()
    factor = frame['reference_adjustment_factor']
    known = np.isfinite(factor) & factor.gt(0)
    assert np.isclose(frame.loc[known, 'adj_close'], frame.loc[known, 'close'] * factor[known],
                      rtol=1e-8, atol=1e-10).all()
    return True

### Controlled repair checks
The following assertions check behavior and accounting, including faults absent from the real database.
Each reference fixture uses a documented adjustment factor of one half. A confirmed-error example
is explicitly artificial; it is not evidence that any real extreme return is wrong.
The price fills and interpolations below do not alter the database.

In [12]:
def repair_fixture(ticker='DEMO'):
    """Five valid trading observations, raw close 10–14 and reference adjustment factor 0.5."""
    result = example_frame(5)
    result['ticker'] = ticker
    result['close'] = np.arange(10.0, 15.0)
    result['open'] = result['close']
    result['low'] = result['close'] - 3
    result['high'] = result['close'] + 3
    result['adj_close'] = result['close'] * 0.5
    return result

def run_repair_example(source, calendar=None, evidence=None):
    """Run one independent example, validating retained rows and the source-row partition."""
    calendar = repair_fixture() if calendar is None else calendar
    result = clean_equity_data(source, prepare_frame(calendar), evidence)
    assert validate_cleaned(result['cleaned'])
    assert len(result['cleaned']) + len(result['removed']) + len(result['quarantined']) == len(source)
    assert result['ledger'].source_row.is_unique and result['ledger'].disposition.ne('pending').all()
    return result

base = repair_fixture()
example_results = []
for column in RAW_PRICE_COLUMNS:
    missing = base.copy()
    missing.loc[2, column] = np.nan
    result = run_repair_example(missing)
    repaired_row = result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[2, 'ts']))].iloc[0]
    assert repaired_row[column] == base.loc[1, column] and repaired_row[column + '_forward_filled']
    example_results.append({'case': f'missing {column}', 'result': 'within-ticker forward fill; bounds valid'})

# No cross-ticker fill: ALT's first opening price has no ALT predecessor.
other = repair_fixture('ALT')
other.loc[0, 'open'] = np.nan
multi_source = pd.concat([base, other], ignore_index=True)
multi_reference = pd.concat([base, repair_fixture('ALT')], ignore_index=True)
result = run_repair_example(multi_source, multi_reference)
assert len(result['quarantined']) == 1 and result['quarantined'].iloc[0].ticker == 'ALT'
example_results.append({'case': 'leading gap / another ticker available', 'result': 'quarantined; no cross-ticker borrowing'})

all_missing = base.copy()
all_missing.loc[2, RAW_PRICE_COLUMNS] = np.nan
result = run_repair_example(all_missing)
assert len(result['removed']) == 1 and result['removed'].iloc[0].reason == 'all_raw_prices_missing_before_fill'
assert len(result['cleaned']) == 4
example_results.append({'case': 'all raw prices missing', 'result': 'removed before any fill'})

for value, expected_reason in [(np.nan, 'missing_or_invalid_adjusted_close'),
                               (np.inf, 'missing_or_invalid_adjusted_close'),
                               (1000.0, 'reference_factor_mismatch')]:
    bad_adjustment = base.copy()
    bad_adjustment.loc[2, 'adj_close'] = value
    result = run_repair_example(bad_adjustment)
    row = result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[2, 'ts']))].iloc[0]
    assert row.adj_close == 6.0 and row.adj_close_repaired and row.adj_close_repair_reason == expected_reason
    example_results.append({'case': f'adjusted close {value}', 'result': 'reconstructed with exact-date factor'})

missing_factor = base.copy()
missing_factor.loc[2, 'adj_close'] = np.nan
result = run_repair_example(missing_factor, base.drop(index=2))
assert len(result['quarantined']) == 1
assert result['quarantined'].iloc[0].reason == 'required_reference_adjustment_factor_unavailable'
example_results.append({'case': 'missing adjusted price and no reference factor', 'result': 'quarantined; no factor carry-forward'})

volume = base.copy()
volume.loc[1, 'volume'] = np.nan
volume.loc[2, 'volume'] = -1
volume.loc[3, 'volume'] = 0
result = run_repair_example(volume)
filled_volume = result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[1, 'ts']))].iloc[0]
assert filled_volume.volume == 0 and filled_volume.volume_missing_filled
assert len(result['quarantined']) == 1 and result['quarantined'].iloc[0].reason == 'invalid_volume'
assert result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[3, 'ts'])), 'zero_volume_review'].all()
example_results.append({'case': 'missing / negative / zero volume', 'result': 'fill with flag / quarantine / retain with review'})

bounds = base.copy()
bounds.loc[2, 'close'] = 500
result = run_repair_example(bounds)
row = result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[2, 'ts']))].iloc[0]
assert row.close == 11 and row.adj_close == 5.5 and row.close_forward_filled
assert row.close_outside_bounds_original
# A forward fill that does not fit today's range must fail validation and be quarantined.
unfillable = base.copy()
unfillable.loc[2, ['low', 'high', 'close', 'adj_close', 'open']] = [50, 60, 55, 27.5, np.nan]
result = run_repair_example(unfillable)
assert len(result['quarantined']) == 1
example_results.append({'case': 'OHLC violations / bad forward fill', 'result': 'repair only if valid; otherwise quarantine'})

# Normalize before deduplicating; conflicting nonidentical copies are not resolved arbitrarily.
identical = base.iloc[[2]].copy()
identical['ticker'] = '<b> demo </b>'
result = run_repair_example(pd.concat([base, identical], ignore_index=True))
assert len(result['removed']) == 1 and len(result['cleaned']) == 5
conflicting = base.iloc[[2]].copy()
conflicting['volume'] = 2000
result = run_repair_example(pd.concat([base, conflicting], ignore_index=True))
assert len(result['quarantined']) == 2 and len(result['cleaned']) == 4
example_results.append({'case': 'formatted duplicate / conflicting key', 'result': 'remove redundant copy / quarantine both versions'})

# Confirmed internal error interpolates to 12; endpoint errors have no two-sided support.
confirmed = base.copy()
confirmed.loc[2, 'close'] = 500
artificial_evidence = pd.DataFrame([{
    'ticker': 'DEMO', 'ts': base.loc[2, 'ts'], 'evidence': 'Controlled fixture: known close was overwritten with 500.'
}])
result = run_repair_example(confirmed, evidence=artificial_evidence)
row = result['cleaned'].loc[lambda x: x.ts.eq(pd.Timestamp(base.loc[2, 'ts']))].iloc[0]
assert row.close == 12 and row.adj_close == 6 and row.close_error_interpolated
for endpoint in [0, 4]:
    endpoint_error = base.copy()
    endpoint_error.loc[endpoint, 'close'] = 500
    evidence = pd.DataFrame([{'ticker': 'DEMO', 'ts': base.loc[endpoint, 'ts'],
                              'evidence': 'Controlled fixture: confirmed endpoint error.'}])
    result = run_repair_example(endpoint_error, evidence=evidence)
    assert len(result['quarantined']) == 1
example_results.append({'case': 'confirmed internal / endpoint close error', 'result': 'interpolate / quarantine without extrapolation'})

# Deleting a date detects a gap but does not manufacture a price observation.
absent = base.drop(index=2)
result = run_repair_example(absent)
missing_report, _ = detect_missing_sessions(result['cleaned'], prepare_frame(base))
assert len(missing_report) == 1 and len(result['cleaned']) == 4
example_results.append({'case': 'absent session', 'result': 'reported; no fabricated row'})
display(pd.DataFrame(example_results))
print('All controlled repair and row-accounting checks passed.')

,case,result
0,missing open,within-ticker forward fill; bounds valid
1,missing high,within-ticker forward fill; bounds valid
2,missing low,within-ticker forward fill; bounds valid
3,missing close,within-ticker forward fill; bounds valid
4,leading gap / another ticker available,quarantined; no cross-ticker borrowing
5,all raw prices missing,removed before any fill
6,adjusted close nan,reconstructed with exact-date factor
7,adjusted close inf,reconstructed with exact-date factor
8,adjusted close 1000.0,reconstructed with exact-date factor
9,missing adjusted price and no reference factor,quarantined; no factor carry-forward


All controlled repair and row-accounting checks passed.


### Apply the documented rules to the actual data
No actual raw-close error has been confirmed, so the real run supplies no interpolation evidence.
The return review and factor comparison above are retained as the pre-repair diagnosis.
Every input row must appear once in the retained/removed/quarantined accounting.

In [13]:
cleaning_result = clean_equity_data(raw, reference_data)
cleaned = cleaning_result['cleaned']
quarantined = cleaning_result['quarantined']
removed = cleaning_result['removed']
row_ledger = cleaning_result['ledger']
assert validate_cleaned(cleaned)
assert len(cleaned) + len(removed) + len(quarantined) == len(raw)
assert set(row_ledger.source_row) == set(raw.source_row)
assert row_ledger.source_row.is_unique
assert set(cleaned.source_row).isdisjoint(set(removed.source_row))
assert set(cleaned.source_row).isdisjoint(set(quarantined.source_row))
assert set(removed.source_row).isdisjoint(set(quarantined.source_row))
display(row_ledger.groupby(['disposition', 'reason']).size().rename('rows').reset_index())
print('Input rows:', len(raw), '| retained:', len(cleaned), '| removed:', len(removed), '| quarantined:', len(quarantined))
display(quarantined[['source_row', 'ticker', 'ts', 'original_open', 'low', 'high', 'close', 'reason']])
repair_flags = ['ticker_normalized', 'open_forward_filled', 'high_forward_filled', 'low_forward_filled',
                'close_forward_filled', 'close_error_interpolated', 'volume_missing_filled',
                'adj_close_repaired', 'zero_volume_review', 'return_outlier_review']
display(cleaned[repair_flags].sum().rename('retained_rows_flagged').to_frame())
display(cleaned.loc[cleaned.adj_close_repaired].groupby('adj_close_repair_reason').size().rename('rows').to_frame())
display(cleaned.loc[cleaned.adj_close_repaired, [
    'ticker', 'ts', 'original_adj_close', 'close', 'reference_adjustment_factor', 'adj_close', 'adj_close_repair_reason'
]].head(10))

,disposition,reason,rows
0,quarantined,unresolved_raw_price_or_ohlc_bounds,5
1,removed,redundant_exact_duplicate,29460
2,retained,passed_final_validation,368246


Input rows: 397711 | retained: 368246 | removed: 29460 | quarantined: 5


,source_row,ticker,ts,original_open,low,high,close,reason
29460,287344,CVX,1962-01-02,0.0,3.24000,3.30000,3.30000,unresolved_raw_price_or_ohlc_bounds
29461,350809,JNJ,1962-01-02,0.0,0.22222,0.22338,0.22338,unresolved_raw_price_or_ohlc_bounds
29462,224791,MCD,1966-07-05,0.0,0.26749,0.27366,0.26955,unresolved_raw_price_or_ohlc_bounds
29463,364569,MRK,1962-01-02,0.0,0.39317,0.39703,0.39537,unresolved_raw_price_or_ohlc_bounds
29464,99638,PG,1962-01-02,0.0,1.42000,1.44000,1.43000,unresolved_raw_price_or_ohlc_bounds


,retained_rows_flagged
ticker_normalized,36985.0
open_forward_filled,0.0
high_forward_filled,0.0
low_forward_filled,0.0
close_forward_filled,0.0
close_error_interpolated,0.0
volume_missing_filled,0.0
adj_close_repaired,28791.0
zero_volume_review,74.0
return_outlier_review,5057.0


,rows
adj_close_repair_reason,
missing_or_invalid_adjusted_close,18374
reference_factor_mismatch,10417


,ticker,ts,original_adj_close,close,reference_adjustment_factor,adj_close,adj_close_repair_reason
17,AAPL,1981-01-08,NaN,0.13505,0.770826,0.104100,missing_or_invalid_adjusted_close
20,AAPL,1981-01-13,NaN,0.13616,0.770858,0.104960,missing_or_invalid_adjusted_close
35,AAPL,1981-02-03,1.264587,0.12333,0.770853,0.095069,reference_factor_mismatch
42,AAPL,1981-02-12,NaN,0.11663,0.770872,0.089907,missing_or_invalid_adjusted_close
86,AAPL,1981-04-16,NaN,0.11161,0.770858,0.086035,missing_or_invalid_adjusted_close
93,AAPL,1981-04-28,2.036653,0.12612,0.770854,0.097220,reference_factor_mismatch
101,AAPL,1981-05-08,2.129290,0.12500,0.770878,0.096360,reference_factor_mismatch
108,AAPL,1981-05-19,NaN,0.12277,0.770866,0.094639,missing_or_invalid_adjusted_close
119,AAPL,1981-06-04,NaN,0.14341,0.770936,0.110560,missing_or_invalid_adjusted_close
146,AAPL,1981-07-14,NaN,0.10603,0.770857,0.081734,missing_or_invalid_adjusted_close


### Validate again and compare before/after
The five quarantined leading observations now appear as five deliberately excluded sessions against
the original reference calendar. This is disclosed rather than hiding the difference or inventing observations.
Zero-volume and extreme-return review flags may remain. High-severity price/key failures may not.
The reference-adjustment check is repeated separately because finite positive prices alone missed that corruption.

The outlier threshold is recomputed after quarantine. Its count can change slightly even though
retained raw closes are unchanged: dropping leading observations changes the sample of returns,
and hence the ticker mean and sample standard deviation. This does not turn a statistical flag into
an established error. Before/after flags are both preserved.

In [14]:
after_frame = prepare_frame(cleaned)
after_report, after_sessions, after_returns = build_quality_report(after_frame, reference_data)
after_adjustments = reference_adjustment_data(after_frame, reference_data)
assert not after_adjustments.adjustment_mismatch.any()
assert not after_report.severity.eq('high').any()
comparison_counts = pd.concat([
    quality_report.groupby('issue_type').size().rename('before'),
    after_report.groupby('issue_type').size().rename('after')
], axis=1).fillna(0).astype(int)
display(comparison_counts)
print('Post-cleaning missing reference sessions:', int(after_sessions.missing_sessions.sum()))
display(after_sessions.loc[after_sessions.missing_sessions.gt(0), ['ticker', 'missing_sessions', 'summary']])
display(pd.concat({'source_rows': data[PRICE_COLUMNS + ['volume']].describe(),
                   'retained_rows': cleaned[PRICE_COLUMNS + ['volume']].describe()}, axis=1))
print('Known reference adjustment consistency verified for', int(cleaned.adjustment_consistency_verified.sum()), 'retained rows.')
print('Raw-price invalid rows retained:', int((~raw_bounds_valid(cleaned)).sum()))
print('Duplicate retained ticker/date keys:', int(cleaned.duplicated(['ticker', 'ts']).sum()))
new_return_flags = cleaned.loc[cleaned.return_outlier_after & ~cleaned.return_outlier_before,
                               ['source_row', 'ticker', 'ts', 'close']]
print('New flags after recomputing the sample threshold on retained data:', len(new_return_flags))
display(new_return_flags)


,before,after
issue_type,,
adjustment_factor_mismatch,11265,0
exact_duplicate,29460,0
invalid_open,5,0
missing_adj_close,19855,0
open_below_low,5,0
return_outlier,5056,5057
ticker_formatting,39936,0
zero_volume,81,74
missing_session,0,5


Post-cleaning missing reference sessions: 5


,ticker,missing_sessions,summary
8,CVX,1,Ticker CVX is missing 1 trading days between 1962-01-02 and 2024-12-06.
14,JNJ,1,Ticker JNJ is missing 1 trading days between 1962-01-02 and 2024-10-18.
17,MCD,1,Ticker MCD is missing 1 trading days between 1966-07-05 and 2024-10-18.
19,MRK,1,Ticker MRK is missing 1 trading days between 1962-01-02 and 2024-12-06.
23,PG,1,Ticker PG is missing 1 trading days between 1962-01-02 and 2024-12-06.


source_rows                                               \
                open           high            low          close   
count  397711.000000  397711.000000  397711.000000  397711.000000   
mean       46.285479      46.747660      45.816279      46.292903   
std        67.132594      67.778210      66.477178      67.141879   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.209938       4.240000       4.160000       4.200000   
50%        19.850000      20.060000      19.630000      19.860000   
75%        57.530000      58.130000      56.950000      57.530000   
max       604.260000     608.630000     597.880000     605.400000   

                                    retained_rows                 \
           adj_close        volume           open           high   
count  377856.000000  3.977110e+05  368246.000000  368246.000000   
mean       51.652250  3.097773e+07      46.309394      46.771824   
std       194.584045  1.266237e+08      67.143254      67.789319   
min         0.002945  0.000000e+00       0.005208       0.005208   
25%         1.390000  1.730700e+06       4.210000       4.250000   
50%        11.960000  4.648200e+06      19.890000      20.090000   
75%        43.770000  1.102380e+07      57.560000      58.160000   
max     10442.510783  9.276606e+09     604.260000     608.630000   

                                                                  
                 low          close      adj_close        volume  
count  368246.000000  368246.000000  368246.000000  3.682460e+05  
mean       45.840375      46.317325      37.274670  3.095184e+07  
std        66.489450      67.154939      63.587872  1.266153e+08  
min         0.004801       0.005208       0.002945  0.000000e+00  
25%         4.170000       4.210000       1.310000  1.731514e+06  
50%        19.670000      19.910000      11.460000  4.649648e+06  
75%        56.997500      57.570000      41.570000  1.102160e+07  
max       597.880000     605.400000     605.400000  9.276606e+09

Known reference adjustment consistency verified for 368246 retained rows.
Raw-price invalid rows retained: 0
Duplicate retained ticker/date keys: 0
New flags after recomputing the sample threshold on retained data: 1


,source_row,ticker,ts,close
219181,258219,MCD,1989-06-21,7.06


### Save `ohlc_hw_cleaned` in the working database
Save dates as ISO `YYYY-MM-DD`. Repair flags, original numeric values, original labels, source-row
identifiers and the adjustment factor travel with every retained row.

Build and validate a staging table first, then replace the cleaned table in a transaction.
This makes reruns reproducible without modifying `ohlc_hw`, `ohlc`, or `market_caps`.
A source-table fingerprint is checked before and after writing. The distributed database outside
this homework directory is never opened for writing.

In [15]:
def frame_fingerprint(frame):
    """Fingerprint values, column order and dtypes in a deterministically ordered table read."""
    digest = hashlib.sha256()
    digest.update(repr(list(zip(frame.columns, map(str, frame.dtypes)))).encode())
    digest.update(pd.util.hash_pandas_object(frame, index=False).to_numpy().tobytes())
    return digest.hexdigest()

stored = cleaned.copy()
stored['ts'] = stored['ts'].dt.strftime('%Y-%m-%d')
for column in stored.select_dtypes(include='bool').columns:
    stored[column] = stored[column].astype('int64')
assert stored['ts'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all()
raw_fingerprint = frame_fingerprint(raw)
reference_fingerprint = frame_fingerprint(reference)
write_connection = sqlite3.connect(DB_PATH)
try:
    current_raw = pd.read_sql_query('SELECT rowid AS source_row, * FROM ohlc_hw ORDER BY rowid', write_connection)
    current_reference = pd.read_sql_query('SELECT * FROM ohlc', write_connection)
    assert frame_fingerprint(current_raw) == raw_fingerprint, 'Source changed since loading; stop.'
    assert frame_fingerprint(current_reference) == reference_fingerprint
    stored.to_sql('ohlc_hw_cleaned_staging', write_connection, if_exists='replace', index=False)
    staged = pd.read_sql_query('SELECT * FROM ohlc_hw_cleaned_staging ORDER BY ticker, ts', write_connection)
    staged['ts'] = pd.to_datetime(staged['ts'], format='%Y-%m-%d', errors='raise')
    assert len(staged) == len(cleaned) and validate_cleaned(staged)
    with write_connection:
        write_connection.execute('BEGIN IMMEDIATE')
        write_connection.execute('DROP TABLE IF EXISTS ohlc_hw_cleaned')
        write_connection.execute('ALTER TABLE ohlc_hw_cleaned_staging RENAME TO ohlc_hw_cleaned')
        write_connection.execute('CREATE UNIQUE INDEX ohlc_hw_cleaned_ticker_ts ON ohlc_hw_cleaned(ticker, ts)')
    raw_after = pd.read_sql_query('SELECT rowid AS source_row, * FROM ohlc_hw ORDER BY rowid', write_connection)
    reference_after = pd.read_sql_query('SELECT * FROM ohlc', write_connection)
    assert frame_fingerprint(raw_after) == raw_fingerprint
    assert frame_fingerprint(reference_after) == reference_fingerprint
    final_count = write_connection.execute('SELECT COUNT(*) FROM ohlc_hw_cleaned').fetchone()[0]
    assert final_count == len(cleaned)
finally:
    write_connection.close()
print(f'Saved and read-back validated {final_count:,} rows in ohlc_hw_cleaned.')
print('Original ohlc_hw and ohlc fingerprints unchanged.')
print('All source rows accounted for; original values and repair/review flags retained.')

Saved and read-back validated 368,246 rows in ohlc_hw_cleaned.
Original ohlc_hw and ohlc fingerprints unchanged.
All source rows accounted for; original values and repair/review flags retained.


### Interpretation and limits
The supplied reference supports consistent date-specific adjustments, but cannot establish the true
cause of every extreme return. Those returns remain available and flagged for review, as do original
zero volumes. Five leading observations are explicitly excluded, and stock histories still end on
different dates. Later reports must disclose their actual coverage and must not extrapolate prices.

Only Part 1 is implemented here. Parts 2–3 will use `ohlc_hw_cleaned`, adjusted-close log returns and
explicit historical report windows; they must not reuse the raw-close diagnostic return convention.